# Strong DiFuMo GAT Colab Experiment

Runs speed-focused DiFuMo GAT/GATv2 graph experiments on CUDA/AMP while keeping cheap DiFuMo MLP and DeepSet controls in the same comparison table. Artifact-only comparisons live in `experiments/difumo_compare_colab.ipynb`.

In [ ]:
import os, platform, subprocess, sys
print('Python:', sys.version)
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('bf16:', torch.cuda.is_bf16_supported())
except Exception as exc:
    print('Torch check failed:', exc)
print('Platform:', platform.platform())

In [ ]:
# Colab dependency install. Runtime -> Change runtime type -> A100 GPU is recommended.
!pip -q install nilearn nibabel scipy huggingface-hub safetensors adapters transformers pyarrow matplotlib pandas scikit-learn tqdm umap-learn torch-geometric

In [ ]:
# Repo clone/pull for Colab. Skip this cell when running inside an existing checkout.
REPO_URL = 'https://github.com/neurovlm/neurovlm.git'
REPO_BRANCH = 'neurovlm_gnn'
REPO_DIR = '/content/neurovlm_gnn'
if not os.path.exists(REPO_DIR):
    !git clone --branch $REPO_BRANCH --single-branch $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin $REPO_BRANCH
    !git -C $REPO_DIR checkout $REPO_BRANCH
    !git -C $REPO_DIR pull --ff-only origin $REPO_BRANCH
os.chdir(REPO_DIR)
!pip -q install -e '.[viz,notebook,metrics]'
print('Working directory:', os.getcwd())

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configure Runs

Everything important is written under a model-scoped Drive tree: checkpoints, run configs, graph stats, metrics, plots, diagnostics, comparison tables, manifests, and the DiFuMo coefficient caches. This notebook uses `runs_difumo_gat`, `data_difumo_gat`, and `checkpoints_difumo_gat` so it can live next to the ALE/Coord notebooks without sharing output folders. If you have a real HCP DiFuMo-space FC matrix, upload it to Drive and set `HCP_FC_PATH`.


In [ ]:
from pathlib import Path
import os, time

MODEL_NAME = 'difumo_gat'
DRIVE_ROOT = '/content/drive/MyDrive/neurovlm_difumo_gat'
DRIVE_DATA_DIR = f'{DRIVE_ROOT}/data_{MODEL_NAME}'
RUN_ROOT = f'{DRIVE_ROOT}/runs_{MODEL_NAME}'
RAW_COEFF_CACHE = f'{DRIVE_DATA_DIR}/difumo512_pubmed_flatmap_coeffs.npz'
ALE_COEFF_CACHE = f'{DRIVE_DATA_DIR}/difumo512_pubmed_ale_fwhm9_coeffs.npz'
COMPARISON_FILE = f'{RUN_ROOT}/difumo_gat_comparison.csv'
EVAL_RESOURCE_DIR = '/content/drive/MyDrive/neurovlm_evaluation_resources'
PRETRAINED_BASELINE_CANDIDATES = [
    'experiments/runs/neurovlm_pubmed_semantic_baseline/main_comparison_summary_row.json',
    f'{EVAL_RESOURCE_DIR}/neurovlm_pubmed_semantic_baseline/main_comparison_summary_row.json',
]
# To resume a previous interrupted batch, set this to that folder name, e.g. '20260509_225451'.
RESUME_BATCH_NAME = None
RUN_BATCH_NAME = RESUME_BATCH_NAME or time.strftime('%Y%m%d_%H%M%S')
HCP_FC_PATH = ''  # e.g. f'{DRIVE_DATA_DIR}/hcp_difumo512_fc.npy'

os.makedirs(RUN_ROOT, exist_ok=True)
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(EVAL_RESOURCE_DIR, exist_ok=True)

# RUN_MODE controls how much work this notebook launches.
# quick: recommended GAT + one cheap control. focused: a compact model/control panel.
# full: all speed-sweep configs; select explicitly before running large sweeps.
RUN_MODE = 'focused'  # one of {'quick', 'focused', 'full'}
RUN_SEMANTIC_EVAL = RUN_MODE != 'quick'

# The backend uses a shared static DiFuMo graph template and flat [B, 512, F]
# tensors for graph models, avoiding fresh PyG collation of repeated graphs.
GAT_BATCH_SIZE = 32
CONTROL_BATCH_SIZE = 512
SHOW_COMMAND = False
FORCE_RERUN = False
STOP_ON_FAILURE = True

BASE = dict(
    epochs=80,
    batch_size=GAT_BATCH_SIZE,
    hidden=64,
    heads=4,
    layers=2,
    dropout=0.2,
    lr_gat=3e-4,
    lr_proj=3e-5,
    temperature=0.07,
    edge_threshold=98,
    early_stopping_patience=12,
    early_stopping_monitor='train_loss',
    early_stopping_min_delta=1e-4,
    val_interval=1,
    coeff_source='flatmap',
    coeff_cache_file=RAW_COEFF_CACHE,
    amp=True,
    umap=True,
)

RECOMMENDED_GAT = dict(
    BASE,
    name='gatv2_coactivation_e98_l2_h4_hidden64',
    model='gat',
    conv='gatv2',
    graph_type='coactivation',
    edge_threshold=98,
    layers=2,
    heads=4,
    hidden=64,
)

CHEAP_CONTROLS = [
    dict(BASE, name='difumo_mlp_baseline', model='mlp', batch_size=CONTROL_BATCH_SIZE, graph_type='no_edge'),
    dict(BASE, name='difumo_deepset_no_edge', model='deepset', batch_size=CONTROL_BATCH_SIZE, graph_type='no_edge', add_centroids=True),
]

FAST_GRAPH_BASELINES = [
    dict(BASE, name='graphsage_coactivation_e98_l2_hidden64', model='sage', graph_type='coactivation', edge_threshold=98, layers=2, hidden=64),
    dict(BASE, name='gcn_coactivation_e98_l2_hidden64', model='gcn', graph_type='coactivation', edge_threshold=98, layers=2, hidden=64),
]

FOCUSED_CONFIGS = [
    RECOMMENDED_GAT,
    dict(RECOMMENDED_GAT, name='gatv2_combined_e98_l2_h4_hidden64', graph_type='combined', add_centroids=True),
    dict(RECOMMENDED_GAT, name='gatv2_shuffled_control_e98_l2_h4_hidden64', graph_type='shuffled'),
    FAST_GRAPH_BASELINES[0],
    *CHEAP_CONTROLS,
]

FULL_GAT_SWEEP = [
    dict(
        BASE,
        name=f'gatv2_coactivation_e{edge_threshold}_l2_h{heads}_hidden{hidden}',
        model='gat',
        conv='gatv2',
        graph_type='coactivation',
        edge_threshold=edge_threshold,
        layers=2,
        heads=heads,
        hidden=hidden,
    )
    for edge_threshold in [95, 98, 99]
    for heads in [2, 4]
    for hidden in [32, 64]
]

CONFIGS_BY_MODE = {
    'quick': [RECOMMENDED_GAT, CHEAP_CONTROLS[0]],
    'focused': FOCUSED_CONFIGS,
    'full': FULL_GAT_SWEEP + FAST_GRAPH_BASELINES + CHEAP_CONTROLS,
}
if HCP_FC_PATH:
    hcp_configs = [
        dict(RECOMMENDED_GAT, name='gatv2_hcp_fc_e98_l2_h4_hidden64', graph_type='hcp_fc', hcp_fc_path=HCP_FC_PATH),
        dict(RECOMMENDED_GAT, name='gatv2_hcp_spatial_combined_e98_l2_h4_hidden64', graph_type='combined', hcp_fc_path=HCP_FC_PATH, add_centroids=True),
    ]
    CONFIGS_BY_MODE['focused'] = CONFIGS_BY_MODE['focused'] + hcp_configs[:1]
    CONFIGS_BY_MODE['full'] = CONFIGS_BY_MODE['full'] + hcp_configs

if RUN_MODE not in CONFIGS_BY_MODE:
    raise ValueError(f'RUN_MODE must be one of {sorted(CONFIGS_BY_MODE)}, got {RUN_MODE!r}')
RUN_CONFIGS = CONFIGS_BY_MODE[RUN_MODE]
print('Drive root:', DRIVE_ROOT)
print('Drive data directory:', DRIVE_DATA_DIR)
print('Run root:', RUN_ROOT)
print('Comparison file:', COMPARISON_FILE)
print('Evaluation resource directory:', EVAL_RESOURCE_DIR)
print('Raw coefficient cache:', RAW_COEFF_CACHE)
print('ALE-smoothed coefficient cache:', ALE_COEFF_CACHE)
print('Pretrained baseline candidates:', PRETRAINED_BASELINE_CANDIDATES)
print('Run mode:', RUN_MODE)
print('Semantic eval:', RUN_SEMANTIC_EVAL)
len(RUN_CONFIGS), [c['name'] for c in RUN_CONFIGS]


In [ ]:
import json, os, shlex, subprocess, sys, time

def cli_key(key):
    return '--' + key.replace('_', '-')

def run_config(cfg):
    name = cfg['name']
    run_dir = f'{RUN_ROOT}/{RUN_BATCH_NAME}/{name}'
    checkpoint_dir = f'{run_dir}/checkpoints_{MODEL_NAME}'
    cmd = [
        sys.executable, 'experiments/train_difumo_gat_colab.py',
        '--run-dir', run_dir,
        '--checkpoint-dir', checkpoint_dir,
        '--comparison-file', COMPARISON_FILE,
    ]
    if RUN_SEMANTIC_EVAL:
        cmd.extend(['--semantic-eval', '--eval-resource-dir', EVAL_RESOURCE_DIR])
    for key, value in cfg.items():
        if key == 'name' or value is None:
            continue
        if isinstance(value, bool):
            if value:
                cmd.append(cli_key(key))
            continue
        cmd.extend([cli_key(key), str(value)])
    os.makedirs(run_dir, exist_ok=True)
    log_path = f'{run_dir}/subprocess.log'
    done_path = f'{run_dir}/eval_results.json'
    ckpt_path = f'{checkpoint_dir}/best_difumo_gat.pt'
    print(f'\n=== {name} ===')
    print('Run dir:', run_dir)
    print('Log file:', log_path)
    if not FORCE_RERUN and os.path.exists(done_path) and os.path.exists(ckpt_path):
        print('Already completed; skipping. Set FORCE_RERUN=True to train again.')
        return {'name': name, 'status': 'skipped', 'run_dir': run_dir}
    if globals().get('SHOW_COMMAND', False):
        print('$', ' '.join(shlex.quote(x) for x in cmd))
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    started = time.time()
    tail = []
    with open(log_path, 'w') as log:
        proc = subprocess.Popen(
            [cmd[0], '-u', *cmd[1:]],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
            tail.append(line.rstrip())
            tail = tail[-80:]
        return_code = proc.wait()
    elapsed_min = (time.time() - started) / 60
    if return_code != 0:
        print(f'\n{name} failed after {elapsed_min:.1f} min. Last log lines:')
        print('\n'.join(tail[-60:]))
        result = {'name': name, 'status': 'failed', 'run_dir': run_dir, 'log_path': log_path, 'return_code': return_code}
        if STOP_ON_FAILURE:
            raise RuntimeError(f'{name} failed with exit code {return_code}. Full log: {log_path}')
        return result
    print(f'{name} finished in {elapsed_min:.1f} min')
    manifest = {
        'name': name,
        'drive_run_dir': run_dir,
        'best_checkpoint': f'{checkpoint_dir}/best_difumo_gat.pt',
        'last_checkpoint': f'{checkpoint_dir}/last_difumo_gat.pt',
        'eval_results': f'{run_dir}/eval_results.json',
        'training_history': f'{run_dir}/training_history.json',
        'graph_stats': f'{run_dir}/graph_stats.json',
        'config': f'{run_dir}/config.json',
        'artifact_manifest': f'{run_dir}/artifacts_manifest.json',
        'comparison_file': COMPARISON_FILE,
        'main_comparison_summary_row': f'{run_dir}/main_comparison_summary_row.csv',
        'semantic_evaluation_manifest': f'{run_dir}/semantic_evaluation_manifest.json',
        'coefficient_cache': cfg.get('coeff_cache_file'),
        'coefficient_source': cfg.get('coeff_source'),
    }
    with open(f'{run_dir}/colab_drive_manifest.json', 'w') as f:
        json.dump(manifest, f, indent=2)
    print('Saved Drive artifacts under:', run_dir)
    return {'name': name, 'status': 'completed', 'run_dir': run_dir}

run_results = []
for cfg in RUN_CONFIGS:
    run_results.append(run_config(cfg))
run_results

In [ ]:
# Summarize results across controls.
import json, os
import pandas as pd
rows = []
for baseline_path in [Path(p) for p in PRETRAINED_BASELINE_CANDIDATES]:
    if not baseline_path.exists():
        continue
    baseline = json.loads(baseline_path.read_text())
    rows.append({
        'run': baseline.get('model', 'pretrained_neurovlm_mlp'),
        'model': 'pretrained_neurovlm_mlp',
        'graph': 'none',
        'test_paper_recall_curve_auc': baseline.get('exact_pmid_paper_recall_curve_auc'),
        'val_paper_recall_curve_auc': None,
        'val_recall@10': None,
        'auc': baseline.get('exact_pmid_paper_recall_curve_auc'),
        'r@1': baseline.get('exact_pmid_recall@1'),
        'r@5': baseline.get('exact_pmid_recall@5'),
        'r@10': baseline.get('exact_pmid_recall@10'),
        'r@50': baseline.get('exact_pmid_recall@50'),
        'mrr': baseline.get('exact_pmid_mrr'),
        'median_rank': baseline.get('exact_pmid_median_rank'),
        'random_r@10': None,
        'network_accuracy': baseline.get('network_accuracy'),
        'mesh_recall@10': baseline.get('mesh_recall@10'),
        'semantic_recall@10': baseline.get('semantic_recall@10'),
        'edges': 0,
        'avg_degree': 0,
        'components': None,
        'epoch_time_sec_mean': None,
        'peak_vram_mb': None,
    })
    print('Loaded pretrained NeuroVLM MLP baseline:', baseline_path)
    break
for cfg in RUN_CONFIGS:
    run_dir = Path(RUN_ROOT) / RUN_BATCH_NAME / cfg['name']
    metrics_path = run_dir / 'eval_results.json'
    graph_path = run_dir / 'graph_stats.json'
    if not metrics_path.exists():
        continue
    eval_results = json.loads(metrics_path.read_text())
    metrics = eval_results['test']
    val_metrics = eval_results.get('val', {})
    graph = json.loads(graph_path.read_text()) if graph_path.exists() else {}
    history_path = run_dir / 'training_history.json'
    history = json.loads(history_path.read_text()) if history_path.exists() else {}
    sem_path = run_dir / 'main_comparison_summary_row.json'
    sem = json.loads(sem_path.read_text()) if sem_path.exists() else {}
    rows.append({
        'run': cfg['name'],
        'model': cfg.get('model'),
        'graph': cfg.get('graph_type', 'none'),
        'test_paper_recall_curve_auc': metrics['mean_auc'],
        'val_paper_recall_curve_auc': val_metrics.get('mean_auc'),
        'val_recall@10': val_metrics.get('mean_recall@10'),
        'auc': metrics['mean_auc'],
        'r@1': metrics['mean_recall@1'],
        'r@5': metrics['mean_recall@5'],
        'r@10': metrics['mean_recall@10'],
        'r@50': metrics['mean_recall@50'],
        'mrr': metrics['mean_mrr'],
        'median_rank': metrics['mean_median_rank'],
        'random_r@10': metrics['mean_random_recall@10'],
        'network_accuracy': sem.get('network_accuracy'),
        'mesh_recall@10': sem.get('mesh_recall@10'),
        'semantic_recall@10': sem.get('semantic_recall@10'),
        'edges': graph.get('number_of_edges'),
        'avg_degree': graph.get('average_degree_directed'),
        'components': graph.get('connected_components'),
        'epoch_time_sec_mean': sum(history.get('epoch_time_sec', [])) / max(1, len(history.get('epoch_time_sec', []))),
        'peak_vram_mb': max(history.get('peak_vram_mb', [0.0]) or [0.0]),
    })
summary = pd.DataFrame(rows).sort_values('auc', ascending=False)
display(summary)
summary_path = Path(RUN_ROOT) / RUN_BATCH_NAME / 'summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved summary:', summary_path)
print('Aggregate comparison CSV:', COMPARISON_FILE)

In [ ]:
# Display plots from the current best run.
from IPython.display import Image, display
if len(summary):
    best = summary.iloc[0]['run']
    print('Best run:', best)
    for png in ['training_curves.png', 'recall_curve.png', 'umap_diagnostics.png']:
        path = Path(RUN_ROOT) / RUN_BATCH_NAME / best / png
        if path.exists():
            display(Image(str(path)))
    for json_name in ['config.json', 'graph_stats.json']:
        path = Path(RUN_ROOT) / RUN_BATCH_NAME / best / json_name
        if path.exists():
            print('\n' + json_name)
            print(path.read_text()[:4000])

## Speed Sweep Notes

The default `RUN_MODE='focused'` starts from `edge_threshold=98`, `layers=2`, `heads=4`, `hidden=64`, then compares it to a combined/spatial variant, shuffled-edge control, GraphSAGE, DiFuMo DeepSet, and DiFuMo MLP. Use `RUN_MODE='quick'` for a smoke test and `RUN_MODE='full'` only when intentionally launching the complete speed sweep over `edge_threshold in [95, 98, 99]`, `heads in [2, 4]`, and `hidden in [32, 64]`.

Interpretation: if GAT beats the DiFuMo MLP and DeepSet controls, graph structure is helping. If GraphSAGE/GCN get similar recall with much lower epoch time or VRAM, use them as the practical graph baseline and reserve GATv2 for promising graph definitions. If shuffled/no-edge controls match real-graph GAT, the DiFuMo graph is not adding useful structure.